# Etapa 3 — Exploração do raw layer via DuckDB

Este notebook abre a primeira conversa em SQL com o dataset de cotações da B3 carregado nas Etapas 1 e 2 do projeto. Não há transformação aqui — isso é responsabilidade do dbt na Etapa 4. O que fazemos é **conectar o DuckDB ao raw layer no MinIO via `httpfs`**, criar a view `raw.cotacoes`, e rodar uma sequência de queries exploratórias para responder perguntas básicas: o dado está completo? Como ele se distribui? Existem gaps de pregão? Onde estão os extremos de retorno?

Cada query mora em um arquivo `.sql` separado em `sql/exploratoria/`. O notebook apenas orquestra — lê o arquivo, executa contra o DuckDB e desenha. SQL é cidadão de primeira classe; manter o código em arquivos versionados é mais útil em revisão que despejar tudo dentro de células.

> **Convenção:** este notebook não inventa SQL inline. Se uma query precisa mudar, ela muda no `.sql`, e o notebook só relê. Isso impede que notebook e repo divirjam silenciosamente.

## 1. Setup

Resolve o caminho da raiz do repo (independente de o notebook ter sido aberto a partir de `notebooks/` ou da raiz), importa a função de conexão do pacote `warehouse`, abre o `warehouse.duckdb` e garante que a view `raw.cotacoes` existe. Roda em segundos — não há cópia de dado, só metadado SQL.

In [1]:
import sys
from pathlib import Path

# Garante que a raiz do repo está no sys.path, independente do CWD em que
# o notebook foi aberto (VS Code usa a pasta do notebook; jupyter da raiz
# usa a raiz). Sem isso, `from warehouse...` falha em VS Code.
REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import pandas as pd
import plotly.express as px

from warehouse.conexao import obter_conexao
from warehouse.setup import criar_schema_raw

SQL_DIR = REPO_ROOT / "sql" / "exploratoria"

def carregar_sql(nome: str) -> str:
    """Lê uma query do arquivo .sql versionado. Falha cedo se o nome estiver errado."""
    caminho = SQL_DIR / nome
    if not caminho.exists():
        raise FileNotFoundError(f"SQL não encontrado: {caminho}")
    return caminho.read_text(encoding="utf-8")

con = obter_conexao()
criar_schema_raw(con)
print("Conexão aberta e schema raw pronto.")

Conexão aberta e schema raw pronto.


## 2. Sanity check — cobertura por ticker

Antes de qualquer análise interessante, valida o básico: todos os 6 tickers do escopo (PETR4, VALE3, ITUB4, BBDC4, WEGE3, ABEV3) aparecem e cobrem aproximadamente a mesma janela temporal? Se alguma linha vier com contagem muito abaixo das outras, é gap de ingestão a investigar antes de prosseguir.

Expectativa: ~5 anos × ~250 pregões/ano ≈ 1250 linhas por ticker, com data mínima próxima de cinco anos atrás e máxima próxima de hoje.

In [2]:
df_cobertura = con.execute(carregar_sql("01_contagem_por_ticker.sql")).fetchdf()
df_cobertura

,ticker,pregoes,primeiro_pregao,ultimo_pregao
0,ABEV3,1246,2021-05-20,2026-05-19
1,BBDC4,1246,2021-05-20,2026-05-19
2,ITUB4,1246,2021-05-20,2026-05-19
3,PETR4,1246,2021-05-20,2026-05-19
4,VALE3,1246,2021-05-20,2026-05-19
5,WEGE3,1246,2021-05-20,2026-05-19


## 3. Amplitude e extremos por ano

A amplitude `(maxima - minima) / minima` mede o quanto o preço oscilou *dentro de um único pregão*. Tirar o máximo desse valor por ticker × ano destaca os dias de movimento extremo — divulgação de balanço, choque macro, evento corporativo.

Esse tipo de visão é a primeira pista para questões maiores: a volatilidade é parecida entre setores? Algum ano se destaca para um ticker específico?

In [3]:
df_amplitude = con.execute(carregar_sql("02_amplitude_por_ano.sql")).fetchdf()
df_amplitude.head(20)

,ticker,ano,amplitude_maxima
0,ABEV3,2021,0.061947
1,ABEV3,2022,0.075362
2,ABEV3,2023,0.081295
3,ABEV3,2024,0.053175
4,ABEV3,2025,0.071184
5,ABEV3,2026,0.082262
6,BBDC4,2021,0.081993
7,BBDC4,2022,0.131148
8,BBDC4,2023,0.075594
9,BBDC4,2024,0.123288


In [4]:
# Heatmap: linha = ticker, coluna = ano, célula = amplitude máxima do ano.
pivot_amplitude = df_amplitude.pivot(index="ticker", columns="ano", values="amplitude_maxima")
fig = px.imshow(
    pivot_amplitude,
    color_continuous_scale="Reds",
    aspect="auto",
    labels=dict(x="Ano", y="Ticker", color="Amplitude máxima"),
    title="Amplitude máxima intradiária por ticker e ano",
)
fig.show()

## 4. Volume médio setorial ao longo do tempo

Agrupa volume diário por **setor** (mapeamento `ticker → setor` está hardcoded na CTE do SQL — vira `dim_empresa` na Etapa 4 e o JOIN deixa de ser explícito). Mostra como o interesse de negociação em cada setor evolui mês a mês.

Atenção: ITUB4 + BBDC4 somam no setor Financeiro, então o nível absoluto desse setor é naturalmente maior que os demais — comparar **tendências**, não valores absolutos.

In [5]:
df_volume = con.execute(carregar_sql("03_volume_medio_setorial.sql")).fetchdf()
df_volume["periodo"] = pd.to_datetime(
    df_volume["ano"].astype(str) + "-" + df_volume["mes"].astype(str).str.zfill(2) + "-01"
)
df_volume.head()

,setor,ano,mes,volume_medio,periodo
0,Consumo,2021,05,2.023860e+07,2021-05-01
1,Consumo,2021,06,3.196547e+07,2021-06-01
2,Consumo,2021,07,2.300731e+07,2021-07-01
3,Consumo,2021,08,2.126371e+07,2021-08-01
4,Consumo,2021,09,2.464455e+07,2021-09-01


In [6]:
fig = px.line(
    df_volume,
    x="periodo",
    y="volume_medio",
    color="setor",
    title="Volume médio diário por setor (escala logarítmica)",
    log_y=True,
)
fig.update_layout(xaxis_title="Mês", yaxis_title="Volume médio diário")
fig.show()

## 5. Retornos diários extremos

20 maiores altas e 20 maiores quedas no histórico, calculadas com `LAG(fechamento) OVER (PARTITION BY ticker ORDER BY data)`. Window function clássica — vale ler o SQL com atenção para entender por que `PARTITION BY ticker` é necessário (sem ele, o `LAG` ignoraria a fronteira de ativo e calcularia retornos cross-ticker, sem sentido).

Ao inspecionar os resultados, vale procurar por padrões: a maioria dos extremos é PETR4 (notícia política)? VALE3 aparece concentrada em algum ano específico (Brumadinho, p.ex.)? Anotar a observação na sequência.

In [7]:
df_extremos = con.execute(carregar_sql("04_retorno_diario_window.sql")).fetchdf()
df_extremos

,extremo,ticker,data,fechamento,retorno_pct
0,alta,BBDC4,2025-05-08,15.080000,15.644172
1,alta,ABEV3,2026-05-05,16.650000,15.304710
2,alta,WEGE3,2024-07-31,50.660000,10.466636
3,alta,VALE3,2022-11-11,82.300003,10.395707
4,alta,ABEV3,2021-10-28,16.700001,9.724050
5,alta,WEGE3,2022-10-26,37.980000,8.359488
6,alta,ITUB4,2023-02-08,23.459841,8.268843
7,alta,WEGE3,2021-07-28,37.200001,8.170984
8,alta,WEGE3,2022-02-15,32.880001,8.015770
9,alta,PETR4,2022-10-03,32.180000,7.986581


> **A preencher após inspecionar:** algum extremo corresponde a evento conhecido que você reconhece (PETR4 com troca de presidente, VALE3 com Brumadinho, ITUB4/BBDC4 em pandemia)? Anotar aqui o que chamou atenção.

## 6. Cobertura temporal — gaps de pregão

Identifica `(data, ticker)` que existem em pelo menos uma linha do raw mas não em outro ticker no mesmo dia. Implementação clássica de **anti-join**: produto cartesiano de datas × tickers, `LEFT JOIN` com o raw, filtrar onde o lado direito veio `NULL`.

Causas possíveis para um gap:
- Suspensão de negociação do ativo por evento corporativo (split, fusão, leilão).
- Pregão parcial em véspera de feriado em que o yfinance simplesmente não trouxe linha para algum ticker.
- Falha de ingestão a corrigir.

Se o resultado vier vazio, ótimo: cobertura completa no período.

In [8]:
df_gaps = con.execute(carregar_sql("05_gaps_de_pregao.sql")).fetchdf()
print(f"Gaps encontrados: {len(df_gaps)}")
df_gaps

Gaps encontrados: 0


,data,ticker


## 7. Fechamento

> **A preencher após rodar o notebook:** o que esta exploração ensinou sobre o dataset? Algum achado vai influenciar decisões da Etapa 4 (modelagem dbt)? Algum gap precisa virar issue de ingestão antes de seguir?

Registrar descobertas em `docs/NOTAS.md` → Etapa 3, especialmente sobre o comportamento de `hive_partitioning` (`ano`, `mes`, `dia` apareceram como colunas?) e da extensão `httpfs` (latência percebida nas queries — está usando cache?).

In [9]:
con.close()